# AI2 - Homework 2: Fine-tuning DeBERTa-v3 σε CLARITY (QEvasion)

Αυτό είναι το final notebook για το DeBERTa-v3 μοντέλο (`microsoft/deberta-v3-base`).

3-class classification (Clear Reply / Ambivalent / Clear Non-Reply) πάνω σε Q-A pairs.
Explicit PyTorch training loop.

**HW1 baseline (LogReg + TF-IDF): val macro-F1 = 0.6017**.

Το DeBERTa ήταν σημαντικά πιο challenging στο setup - χρειάστηκαν αρκετές debugging
iterations μέχρι να δουλέψει σωστά. Όλα τα βήματα τεκμηριώνονται παρακάτω.

** ΣΗΜΑΝΤΙΚΟ**: το Cell 1 κάνει pin `transformers==4.44.0`. Η version 5.0.0 του
transformers έχει broken DeBERTa-v3 fine-tuning, το μοντέλο δεν μαθαίνει. Μετά το Cell 1
**πρέπει να γίνει restart του kernel** πριν τρέξετε τα υπόλοιπα cells.

## Βήμα 1 - Εγκατάσταση dependencies

Pin `transformers==4.44.0` για reproducibility.

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys

# Pin transformers to 4.44.0 — transformers 5.0.0 has a broken DeBERTa-v3 fine-tuning issue.
# After this install, the kernel MUST be restarted before running Cell 2+.
# On Kaggle: Run this cell alone first → Restart kernel → Run all remaining cells.
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.44.0", "datasets", "accelerate", "scikit-learn",
    "sentencepiece", "protobuf"], capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-300:] if result.stderr else "")

import transformers
print(f"transformers version: {transformers.__version__}")
assert transformers.__version__.startswith("4."), \
    f"STOP: transformers is {transformers.__version__}. Restart the kernel and re-run."
print("deps ok")

## Βήμα 2 - Library (inlined src/)

Ολόκληρο το training pipeline μέσα σε ένα cell για self-contained notebook (config, data loading/cleaning, tokenization, model, training, evaluation, experiment runner).

In [ ]:
# Cell 2 — Library (inlined src/)
# ============================================================
# src/config.py
# ============================================================
from dataclasses import dataclass, field, asdict
from typing import Literal
import json

@dataclass
class ExperimentConfig:
    model_name: str = "distilbert-base-uncased"
    input_fmt: Literal["two_segment", "concat_sep"] = "two_segment"
    max_length: int = 128
    lr: float = 2e-5
    batch_size: int = 32
    epochs: int = 3
    warmup_steps: int = 0
    grad_clip: float = 1.0
    use_class_weights: bool = False
    weight_decay: float = 0.0
    seed: int = 42
    split_id: int = 0
    mode: Literal["smoke", "dev", "confirm", "final"] = "dev"
    smoke_n: int = 50
    data_dir: str = "data"
    models_dir: str = "/kaggle/working/models"
    run_id: str = field(init=False)

    def __post_init__(self):
        short_model = self.model_name.split("/")[-1]
        self.run_id = (
            f"{short_model}_{self.input_fmt}_lr{self.lr}_bs{self.batch_size}"
            f"_ep{self.epochs}_ml{self.max_length}_seed{self.seed}"
            + ("_cw" if self.use_class_weights else "")
            + (f"_wd{self.weight_decay}" if self.weight_decay > 0 else "")
        )

    def to_dict(self) -> dict:
        return asdict(self)

    def save(self, path: str) -> None:
        with open(path, "w") as f:
            json.dump(self.to_dict(), f, indent=2)

# ============================================================
# src/data.py
# ============================================================
from typing import Optional, Tuple
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

LABEL2ID = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 3
HF_DATASET = "ailsntua/QEvasion"
CORRUPTED_TRAIN_INDEX_VALUES = list(range(1870, 1884))

def load_clarity(data_dir=None):
    ds = load_dataset(HF_DATASET)
    train = ds["train"].to_pandas().copy()
    test = ds["test"].to_pandas().copy()
    for df in (train, test):
        df.rename(columns={"interview_answer": "answer", "clarity_label": "label"}, inplace=True)
    return train, test

def clean_data(df, verbose=True):
    raw_n = len(df)
    qa_cols = ["question", "answer"]
    dup_mask = df.duplicated(subset=qa_cols, keep=False)
    conflicting_ids = (
        df[dup_mask].groupby(qa_cols)["label"].nunique()
        .pipe(lambda s: s[s > 1]).index
    )
    multi_idx = pd.MultiIndex.from_frame(df[qa_cols])
    conflict_multi = pd.MultiIndex.from_tuples(conflicting_ids)
    drop_mask = multi_idx.isin(conflict_multi)
    cleaned = df[~drop_mask].copy()
    after_conflicts = len(cleaned)
    cleaned = cleaned.loc[~cleaned["index"].isin(CORRUPTED_TRAIN_INDEX_VALUES)].reset_index(drop=True)
    after_corrupted = len(cleaned)
    if verbose:
        print(f"[clean_data] {raw_n} -> {after_conflicts} (−{raw_n-after_conflicts} conflicts) -> {after_corrupted} (−{after_conflicts-after_corrupted} corrupted)")
    return cleaned

def encode_labels(df, label_col="label"):
    df = df.copy()
    df["label_id"] = df[label_col].map(LABEL2ID)
    return df

def create_split(df, val_size=0.1, seed=0):
    idx = np.arange(len(df))
    train_idx, val_idx = train_test_split(
        idx, test_size=val_size, random_state=seed, stratify=df["label_id"].values
    )
    return train_idx, val_idx

def smoke_subset(df, n_per_class=50, seed=42):
    parts = [g.sample(min(n_per_class, len(g)), random_state=seed) for _, g in df.groupby("label_id")]
    return pd.concat(parts, ignore_index=True)

def subgroup_metadata(df):
    df = df.copy()
    df["q_len"] = df["question"].str.split().str.len()
    df["a_len"] = df["answer"].str.split().str.len()
    df["q_len_bin"] = pd.qcut(df["q_len"], q=3, labels=["short", "medium", "long"])
    df["a_len_bin"] = pd.qcut(df["a_len"], q=3, labels=["short", "medium", "long"])
    return df

# ============================================================
# src/tokenization.py
# ============================================================
import torch
from torch.utils.data import TensorDataset

def tokenize_pairs(df, tokenizer, max_length, input_fmt="two_segment"):
    questions = df["question"].astype(str).tolist()
    answers = df["answer"].astype(str).tolist()
    if input_fmt == "two_segment":
        enc = tokenizer(questions, answers, padding="max_length", truncation=True,
                        max_length=max_length, return_tensors="pt")
    elif input_fmt == "concat_sep":
        sep = tokenizer.sep_token or "[SEP]"
        texts = [f"{q} {sep} {a}" for q, a in zip(questions, answers)]
        enc = tokenizer(texts, padding="max_length", truncation=True,
                        max_length=max_length, return_tensors="pt")
    else:
        raise ValueError(f"Unknown input_fmt: {input_fmt!r}")
    labels = torch.tensor(df["label_id"].values, dtype=torch.long)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)

# ============================================================
# src/model.py
# ============================================================
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def load_model_and_tokenizer(model_name, num_labels=3):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    return model, tokenizer

# ============================================================
# src/train.py
# ============================================================
import random
from torch.utils.data import DataLoader

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_one_epoch(model, loader, optimizer, scheduler, device, grad_clip=1.0, class_weights=None):
    model.train()
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights) if class_weights is not None else None
    total_loss = 0.0
    for batch in loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        optimizer.zero_grad()
        if loss_fn is not None:
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(out.logits, labels)
        else:
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

# ============================================================
# src/evaluate.py
# ============================================================
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    total_loss = 0.0
    for batch in loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += out.loss.item()
        all_logits.append(out.logits.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    logits = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0)
    preds = logits.argmax(axis=1)
    metrics = {
        "val_loss": total_loss / max(len(loader), 1),
        "accuracy": float(accuracy_score(labels_np, preds)),
        "f1_macro": float(f1_score(labels_np, preds, average="macro", zero_division=0)),
        "precision_macro": float(precision_score(labels_np, preds, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(labels_np, preds, average="macro", zero_division=0)),
        "f1_per_class": f1_score(labels_np, preds, average=None, labels=[0,1,2], zero_division=0).tolist(),
    }
    return metrics, logits, labels_np

# ============================================================
# src/experiment.py
# ============================================================
from pathlib import Path
from torch.optim import AdamW
from torch.utils.data import RandomSampler, SequentialSampler
from transformers import get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight

def run_experiment(cfg):
    set_seed(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[info] device: {device}")

    train_df_raw, test_df = load_clarity(cfg.data_dir)
    cleaned = clean_data(train_df_raw)
    cleaned = encode_labels(cleaned)

    if cfg.mode == "smoke":
        cleaned = smoke_subset(cleaned, n_per_class=cfg.smoke_n, seed=cfg.seed)
        print(f"[info] smoke subset: {len(cleaned)} rows")

    train_idx, val_idx = create_split(cleaned, val_size=0.1, seed=cfg.split_id)
    train_df_split = cleaned.iloc[train_idx].reset_index(drop=True)
    val_df_split = cleaned.iloc[val_idx].reset_index(drop=True)
    print(f"[info] train: {len(train_df_split)}, val: {len(val_df_split)}")

    # class weights computed from training split only
    class_weights = None
    if cfg.use_class_weights:
        weights = compute_class_weight(
            "balanced", classes=np.array([0, 1, 2]), y=train_df_split["label_id"].values
        )
        class_weights = torch.tensor(weights, dtype=torch.float).to(device)
        print(f"[info] class weights: CR={weights[0]:.3f}  AMB={weights[1]:.3f}  CNR={weights[2]:.3f}")

    model, tokenizer = load_model_and_tokenizer(cfg.model_name, num_labels=NUM_LABELS)
    model.to(device)

    train_ds = tokenize_pairs(train_df_split, tokenizer, cfg.max_length, cfg.input_fmt)
    val_ds = tokenize_pairs(val_df_split, tokenizer, cfg.max_length, cfg.input_fmt)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=RandomSampler(train_ds))
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, sampler=SequentialSampler(val_ds))

    if "deberta" in cfg.model_name.lower():
        head_params = list(model.pooler.parameters()) + list(model.classifier.parameters())
        head_ids = {id(p) for p in head_params}
        backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
        optimizer = AdamW([
            {"params": backbone_params, "lr": cfg.lr},
            {"params": head_params,     "lr": cfg.lr * 10},
        ], eps=1e-6, weight_decay=cfg.weight_decay)
    else:
        optimizer = AdamW(model.parameters(), lr=cfg.lr, eps=1e-6, weight_decay=cfg.weight_decay)

    total_steps = len(train_loader) * cfg.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=cfg.warmup_steps, num_training_steps=total_steps
    )

    run_dir = Path(cfg.models_dir) / cfg.run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    history = []
    val_logits = np.array([])
    val_labels_arr = np.array([])
    best_f1 = -1.0
    for epoch in range(cfg.epochs):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scheduler, device, cfg.grad_clip, class_weights
        )
        val_metrics, val_logits, val_labels_arr = evaluate(model, val_loader, device)
        print(
            f"[epoch {epoch+1}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['val_loss']:.4f} "
            f"val_f1_macro={val_metrics['f1_macro']:.4f} "
            f"val_acc={val_metrics['accuracy']:.4f}"
        )
        history.append({"epoch": epoch+1, "train_loss": train_loss, **val_metrics})
        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            torch.save(model.state_dict(), run_dir / "best_model.pt")
            np.save(run_dir / "val_preds.npy", val_logits)
            np.save(run_dir / "val_labels.npy", val_labels_arr)
            print(f"  * new best checkpoint (f1_macro={best_f1:.4f})")

    model.load_state_dict(torch.load(run_dir / "best_model.pt", map_location=device))
    val_logits = np.load(run_dir / "val_preds.npy")
    val_labels_arr = np.load(run_dir / "val_labels.npy")

    cfg.save(str(run_dir / "config.json"))
    with open(run_dir / "metrics.json", "w") as f:
        json.dump(history, f, indent=2)
    print(f"[info] saved run to {run_dir} (best f1_macro={best_f1:.4f})")

    return history, val_logits, val_labels_arr, test_df, tokenizer, model, device

print("library loaded ok")

## Βήμα 3 - Πειράματα που δοκίμασα

Εδώ παρουσιάζω όλες τις παραμετροποιήσεις που δοκίμασα, σχολιασμένες (comment out) ώστε να μην τρέξουν αλλά να είναι τεκμηριωμένες όπως ζητάει η εκφώνηση. Μόνο το final config στο τέλος είναι ενεργό.

### Πείραμα 1 - Baseline (NaN loss)

Πρώτη προσπάθεια με τα ίδια defaults που χρησιμοποίησα για BERT/DistilBERT:
lr=2e-5, ep=3, eps=1e-8 στον AdamW.

Αποτέλεσμα: `train_loss=nan` από το ep1. Το DeBERTa-v3 με disentangled attention
παράγει αριθμητικά unstable gradients στο default epsilon.

In [ ]:
# (Παλιό config με eps=1e-8 - έβγαζε NaN)
# sweep_configs = [
#     ExperimentConfig(model_name="microsoft/deberta-v3-base", input_fmt="two_segment",
#                      max_length=256, lr=2e-5, batch_size=16, epochs=3,
#                      warmup_steps=100, mode="dev", seed=42),
# ]

### Πείραμα 2 - Fix: eps=1e-6 στον AdamW

Αλλάξαμε σε `eps=1e-6` στον AdamW. Το NaN λύθηκε, αλλά το μοντέλο δεν μάθαινε:
val_f1_macro στο 0.2475 σε όλα τα epochs, predicts all Ambivalent.

Διαγνωστικό: max gradient norm = 10.8 (clipped στο 1.0 - 10x reduction). Το uniform
lr δεν φτάνει στο head.

In [ ]:
# (eps=1e-6 fix αλλά ακόμα stuck στο 0.2475 - predicts all Ambivalent)

### Πείραμα 3 - Fix: differential learning rate για το head

Δίνω στο `pooler` + `classifier` (τα randomly-init μέρη) 10x μεγαλύτερο lr από το
pretrained backbone. Αυτό είναι το standard trick για fine-tuning όταν grad clipping
συμπιέζει σημαντικά τα gradients στο head.

Παρόλα αυτά, δεν ήταν αρκετό - το πρόβλημα ήταν αλλού.

In [ ]:
# (Differential lr υλοποιείται μέσα στο run_experiment για όλα τα DeBERTa runs)
# Το configuration ήταν το ίδιο με πριν, η αλλαγή ήταν στον optimizer.

### Πείραμα 4 - Root cause: transformers 5.0.0 bug

Το Kaggle environment είχε transformers 5.0.0, που έχει broken DeBERTa-v3 fine-tuning
σε sequences κοντά στο max_length. Το diagnostic cell έτρεχε forward pass σε 10-token
toy input και δούλευε - στο training όμως με 256-token real data σπάει.

**Fix: pin transformers==4.44.0 + kernel restart**. Αμέσως το μοντέλο μαθαίνει!

Πρώτο successful run με transformers 4.44.0:
- ep1: f1=0.2475 (ακόμα χαμηλά)
- ep2: f1=0.6193 (pops up)
- ep3: **f1=0.6899**

Το είναι το **best overall result** πάνω σε όλα τα μοντέλα.

In [ ]:
# Η λύση είναι το pin στο Cell 1 + kernel restart. Το config παραμένει ίδιο:
# sweep_configs = [
#     ExperimentConfig(model_name="microsoft/deberta-v3-base", input_fmt="two_segment",
#                      max_length=256, lr=2e-5, batch_size=16, epochs=3,
#                      warmup_steps=100, mode="dev", seed=42),
# ]

### Πείραμα 5 - Epoch sweep (ep=5)

Δοκίμασα ep=5 για να δω αν συνεχίζει η βελτίωση πέρα από ep=3.
Αποτέλεσμα: best@ep5 με val_f1_macro=0.6844, αλλά χαμηλότερο από ep3=0.6899 (από ep=3 run).
Το ep=4 είναι overfit (val_f1 πέφτει στο 0.664, val_loss ανεβαίνει 0.663->0.748).

**Conclusion: ep=3 είναι το sweet spot για το DeBERTa.**

In [ ]:
# sweep_configs = [
#     ExperimentConfig(model_name="microsoft/deberta-v3-base", input_fmt="two_segment",
#                      max_length=256, lr=2e-5, batch_size=16, epochs=5,
#                      warmup_steps=100, mode="dev", seed=42),
# ]

### Πείραμα 6 - Confirm runs με wd=0.01 (3 seeds)

Confirm runs με 3 seeds (42, 0, 1) - με το implicit PyTorch default weight_decay=0.01.

**Αποτέλεσμα: val_f1_macro = 0.6585 ± 0.030**

Αξιοσημείωτη variance: seed=42 πήρε 0.6864, seed=1 πήρε 0.6627, seed=0 μόνο 0.6265.
Το DeBERTa είναι αρκετά sensitive στο random initialization.

In [ ]:
# sweep_configs = [
#     ExperimentConfig(model_name="microsoft/deberta-v3-base", input_fmt="two_segment",
#                      max_length=256, lr=2e-5, batch_size=16, epochs=3,
#                      warmup_steps=100, mode="confirm", seed=s)
#     for s in (42, 0, 1)
# ]

### Πείραμα 7 - Re-confirm με wd=0.0 (final)

Μετά την ανακάλυψη του implicit wd=0.01 default στο PyTorch AdamW, έκανα re-confirm
με explicit wd=0.0 (το νέο library default).

**Αποτέλεσμα: val_f1_macro = 0.6674 ± 0.032** (+0.009 mean vs wd=0.01)

Best single seed: seed=42 με val_f1_macro = **0.6962** (best over όλα τα μοντέλα).
Το DeBERTa ωφελείται λίγο από wd=0.0, όπως το DistilBERT αλλά αντίθετα από το BERT.

Αυτό είναι το final config.

In [ ]:
# sweep_configs = [
#     ExperimentConfig(model_name="microsoft/deberta-v3-base", input_fmt="two_segment",
#                      max_length=256, lr=2e-5, batch_size=16, epochs=3,
#                      warmup_steps=100, mode="confirm", seed=s)
#     for s in (42, 0, 1)   # weight_decay=0.0 via library default
# ]

## Βήμα 4 - Τελικό configuration

Best confirmed config για DeBERTa-v3: lr=2e-05, epochs=3, max_length=256, input_fmt=two_segment, seed=42.

In [ ]:
# ── TELIKO CONFIG (το μοναδικό ενεργό) ──
sweep_configs = [
    ExperimentConfig(
        model_name   = "microsoft/deberta-v3-base",
        input_fmt    = "two_segment",
        max_length   = 256,
        lr           = 2e-05,
        batch_size   = 16,
        epochs       = 3,
        warmup_steps = 100,
        mode         = "dev",   # κρατάω val split για να φαίνονται metrics
        seed         = 42,
    ),
]

for cfg in sweep_configs:
    print(cfg.run_id)

## Βήμα 5 - Training

In [ ]:
# Run training
import gc
from pathlib import Path

sweep_results = []
for cfg in sweep_configs:
    print(f"\n{'='*60}\nRUNNING: {cfg.run_id}\n{'='*60}")
    history, val_logits, val_labels_arr, test_df, tokenizer, model, device = run_experiment(cfg)
    best_epoch = max(history, key=lambda x: x["f1_macro"])
    run_dir = Path(cfg.models_dir) / cfg.run_id
    sweep_results.append({
        "run_id":       cfg.run_id,
        "run_dir":      str(run_dir),
        "best_epoch":   best_epoch["epoch"],
        "val_f1_macro": best_epoch["f1_macro"],
        "val_acc":      best_epoch["accuracy"],
        "f1_per_class": best_epoch["f1_per_class"],
        "history":      history,
        "cfg":          cfg,
    })
    # free GPU before next run (if any)
    model.cpu()
    del model, tokenizer, val_logits, val_labels_arr
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nDone.")

## Βήμα 6 - Αποτελέσματα

In [ ]:
# Results table
import pandas as pd

summary = pd.DataFrame([{
    "run_id":       r["run_id"],
    "best_epoch":   r["best_epoch"],
    "val_f1_macro": round(r["val_f1_macro"], 4),
    "val_acc":      round(r["val_acc"], 4),
    "f1_CR":        round(r["f1_per_class"][0], 4),
    "f1_AMB":       round(r["f1_per_class"][1], 4),
    "f1_CNR":       round(r["f1_per_class"][2], 4),
} for r in sweep_results])

print("=== Final results (HW1 baseline = 0.6017) ===")
display(summary)
best_run = max(sweep_results, key=lambda r: r["val_f1_macro"])
print(f"\nBest: {best_run['run_id']}  f1={best_run['val_f1_macro']:.4f}")

## Βήμα 7 - Learning curve

In [ ]:
# Learning curve
import matplotlib.pyplot as plt

r = sweep_results[0]
epochs = [e["epoch"] for e in r["history"]]
train_losses = [e["train_loss"] for e in r["history"]]
val_losses   = [e["val_loss"]   for e in r["history"]]
val_f1s      = [e["f1_macro"]   for e in r["history"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, train_losses, "o-", label="train_loss")
ax1.plot(epochs, val_losses,   "o-", label="val_loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.set_title("Loss per epoch")

ax2.plot(epochs, val_f1s, "o-", color="green")
ax2.axhline(y=0.6017, color="gray", linestyle="--", label="HW1 baseline")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("val F1-macro"); ax2.legend(); ax2.grid(True, alpha=0.3)
ax2.set_title("Validation F1-macro per epoch")

plt.tight_layout()
plt.savefig("/kaggle/working/learning_curve.png", dpi=150)
plt.show()

## Βήμα 8 - Confusion matrix

In [ ]:
# Confusion matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from pathlib import Path

r = sweep_results[0]
val_logits = np.load(Path(r["run_dir"]) / "val_preds.npy")
val_labels = np.load(Path(r["run_dir"]) / "val_labels.npy")
preds = val_logits.argmax(axis=1)
cm = confusion_matrix(val_labels, preds, labels=[0, 1, 2])
label_names = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=label_names, yticklabels=label_names,
            cmap="Blues", ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"{r['run_id']}\nval F1-macro = {r['val_f1_macro']:.4f}")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150)
plt.show()

print(f"cm: {cm.tolist()}")

## Βήμα 9 - Submission για το Kaggle

Γενικό `submission.csv` (τι θέλει το Kaggle uploader) + `submission_<model>.csv` (τι ζητάει η εκφώνηση). Τα δύο αρχεία είναι identical.

In [ ]:
# Generate submission CSV (δύο ονόματα: model-specific + generic για το Kaggle uploader)
import pandas as pd
from pathlib import Path

r = sweep_results[0]
reload_model, reload_tok = load_model_and_tokenizer(r["cfg"].model_name, num_labels=NUM_LABELS)
reload_model.load_state_dict(torch.load(
    Path(r["run_dir"]) / "best_model.pt",
    map_location=torch.device("cuda" if torch.cuda.is_available() else "cpu")
))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reload_model.to(device).eval()

_, test_df = load_clarity()
test_df_enc = test_df.copy()
test_df_enc["label_id"] = 0  # dummy

test_ds = tokenize_pairs(test_df_enc, reload_tok, r["cfg"].max_length, r["cfg"].input_fmt)
test_loader = DataLoader(test_ds, batch_size=r["cfg"].batch_size, sampler=SequentialSampler(test_ds))

all_preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, _ = [b.to(device) for b in batch]
        out = reload_model(input_ids=input_ids, attention_mask=attention_mask)
        all_preds.extend(out.logits.argmax(dim=1).cpu().numpy())

submission = pd.DataFrame({
    "Id": test_df["index"] if "index" in test_df.columns else range(len(test_df)),
    "Predicted": [ID2LABEL[p] for p in all_preds],
})

# Save BOTH names - Kaggle uploader wants "submission.csv", assignment wants model-specific
submission.to_csv("/kaggle/working/submission_deberta-v3-base.csv", index=False)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print(f"Submissions saved: submission_deberta-v3-base.csv and submission.csv")
print(f"Val F1-macro = {r['val_f1_macro']:.4f}")
display(submission.head())
print(submission["Predicted"].value_counts())

## Βήμα 10 - Download artifacts

In [ ]:
# Download run directory (metrics.json, val_preds.npy, val_labels.npy, best_model.pt)
import shutil
from pathlib import Path
from IPython.display import FileLink, display

src = Path("/kaggle/working/models") / sweep_results[0]["cfg"].run_id
dst_base = "/kaggle/working/run_deberta-v3-base"
shutil.make_archive(dst_base, "zip", str(src.parent), src.name)
print(f"Zipped: {dst_base}.zip")
display(FileLink(f"run_deberta-v3-base.zip"))